In [12]:
#Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"

t5_tokenizer = AutoTokenizer.from_pretrained(model_name)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

## Getting baseline ROUGE Score

In [24]:
#Bring in test dataset (which we will make summaries for with untrained T5)
%store -r test_paired_summaries

#Dictinory with the key being the text name and the value being a 3 element tuple of (text, summary, human-made flag) where human-made flag is true if the summary was made by a human



In [25]:
def summarize_story(story):
    prompt = "summarize: "

    # Tokenize the input text
    inputs = t5_tokenizer(prompt+story, max_length=1024, truncation=True, return_tensors="pt")

    # Dynamic max_new_tokens based on input length
    input_len = inputs["input_ids"].shape[1]

    #summary = 30% of input length, minimum of 30 tokens
    max_new = max(30, int(input_len * 0.3))

    # Generate the output sequence
    outputs = t5_model.generate(inputs["input_ids"], max_new_tokens=max_new)

    # Decode the output IDs back into human-readable text
    candidate = t5_tokenizer.batch_decode(outputs, skip_special_tokens=True, clean_up_tokenization_spaces=False)

    return candidate[0]

In [28]:
#dataset which contains scene or sonnet name as key and T5 summary as value
t5_test_sumaries=dict()

#Get summary of all stories in test dataset
for k, v in test_paired_summaries.items():
    story=v[0]
    t5_test_sumaries[k] = summarize_story(story)
    break #for testing


In [29]:
print(t5_test_sumaries)

{'Julius_Caesar-Act I-Scene I': 'MARULLUS. a trade art thou? answer me directly. MARULLUS. a trade, sir, that I hope I may use with a safe conscience . MARULLUS. a trade, sir, that is indeed, sir, a surgeon to old shoes .'}
